# Jupyter Notebook - ChDB
Предварительны анализ файла без загрузки данных в `clickhouse`.<br>
Для работы с файлом используется `ChDb` https://clickhouse.com/docs/chdb

In [1]:
%load_ext autotime
%matplotlib inline

time: 427 ms (started: 2026-06-01 10:04:28 +03:00)


In [2]:
import time

start_time = time.perf_counter()

time: 465 μs (started: 2026-06-01 10:04:28 +03:00)


## Подключаем модули

In [3]:
import socket
from pathlib import Path

import chdb
import pandas as pd
from chdb import datastore as ds
from my_module.my_func import (
    get_null_exists_estimation,
    get_uniq_estimation,
    highlight_values,
)


Default dirty pages decay period: 5000ms


time: 368 ms (started: 2026-06-01 10:04:28 +03:00)


## Загружаем файл

In [ ]:
hostname = socket.gethostname()
in_docker = True
if hostname == "home-NMH-WDX9":
    in_docker = False

if in_docker:
    file = (Path(Path.cwd().parent) / "db/raw_data/cars/cars_sales_zstd.parquet").as_posix()
else:
    file = (
        Path(Path.cwd().parent.parent.parent)
        / "datasets/raw_data/cars/cars_sales_zstd.parquet"
    ).as_posix()

dsf = ds.read_parquet(file)  # type: ignore

time: 1.04 ms (started: 2026-06-01 10:04:29 +03:00)


## Выводим `describe` статистику по выборочным полям

In [5]:
with pd.option_context(
    "display.float_format",
    "{:.2f}".format,
):
    display(dsf.describe())


,year,mileage,power,price
count,724644.00,771799.00,1273353.00,1294757.00
mean,2009.68,154893.40,141.56,1444357.82
std,9.37,100738.34,65.64,1970257.41
min,1936.00,1000.00,1.00,270.00
25%,2003.00,82000.00,98.00,425000.00
50%,2011.00,144000.00,128.00,870000.00
75%,2017.00,211000.00,163.00,1765000.00
max,2023.00,1000000.00,1000.00,150000000.00


time: 272 ms (started: 2026-06-01 10:04:29 +03:00)


## Получаем общее кол-во записей

In [6]:
sql = f"SELECT COUNT(*) FROM file('{file}', 'Parquet');"  # noqa: S608
result = chdb.query(sql)
rows_count = int(result.data())

time: 4.56 ms (started: 2026-06-01 10:04:29 +03:00)


## Формируем отчет по типам, LowCardinality, Not Null, Null
В отчете используется цветовое выделение значений. github markdown это не поддерживает.

In [7]:
sql = f"DESCRIBE TABLE file('{file}', 'Parquet');"
df_table_columns = chdb.query(sql, "DataFrame")

table_columns = df_table_columns["name"].to_list()
df_table_columns.set_index("name", inplace=True)

field_not_null_counts = {}
field_uniq = {}
for column in table_columns:
    sql = f"""SELECT
    uniq(`{column}`) as uniq_count,
    count(*) as non_null_count
    FROM file('{file}', 'Parquet')
    WHERE `{column}` IS NOT NULL;"""  # noqa: S608

    q = chdb.query(sql, "DataFrame")
    field_not_null_counts[column] = q["non_null_count"].iloc[0]
    field_uniq[column] = q["uniq_count"].iloc[0]

df_uniq = pd.DataFrame.from_dict(field_uniq, orient="index", columns=["Uniq Count"])
df_uniq["Uniq Count %"] = round((df_uniq["Uniq Count"] * 100) / rows_count, 2)

df_non_null = pd.DataFrame.from_dict(
    field_not_null_counts, orient="index", columns=["Non-Null Count"]
)
df_non_null["Null Count"] = rows_count - df_non_null["Non-Null Count"]
df_non_null["Null Count %"] = round((df_non_null["Null Count"] * 100) / rows_count, 2)
df_non_null["Description Null"] = df_non_null.apply(
    lambda row: get_null_exists_estimation(row, 30), axis=1
)

df_rez_null_count = (
    pd
    .concat([df_table_columns, df_uniq, df_non_null], axis=1)
    .reset_index()
    .rename(columns={"index": "field"})
)

df_rez_null_count["Description Uniq"] = df_rez_null_count.apply(
    lambda row: get_uniq_estimation(row, 10), axis=1
)

df_rez_null_count = df_rez_null_count[
    [
        "field",
        "type",
        "Uniq Count",
        "Uniq Count %",
        "Description Uniq",
        "Non-Null Count",
        "Null Count",
        "Null Count %",
        "Description Null",
    ]
]

rez_styled = df_rez_null_count.style.apply(
    lambda row: highlight_values(row, threshold_null=30, threshold_uniq=10), axis=1
).format({"Null Count %": "{:.2f}", "Uniq Count %": "{:.2f}"})

rez_styled


,field,type,Uniq Count,Uniq Count %,Description Uniq,Non-Null Count,Null Count,Null Count %,Description Null
0,brand,Nullable(String),160,0.01,LowCardinality,1294757,0,0.00,
1,name,Nullable(String),2223,0.17,LowCardinality,1294757,0,0.00,
2,bodyType,Nullable(String),11,0.00,LowCardinality,1294757,0,0.00,
3,color,Nullable(String),16,0.00,LowCardinality,1257029,37728,2.91,Мало
4,fuelType,Nullable(String),3,0.00,LowCardinality,1289815,4942,0.38,Мало
5,year,Nullable(Float64),78,0.01,,724644,570113,44.03,Много
6,mileage,Nullable(Float64),821,0.06,,771799,522958,40.39,Много
7,transmission,Nullable(String),5,0.00,LowCardinality,1289563,5194,0.40,Мало
8,power,Nullable(Float64),541,0.04,,1273353,21404,1.65,Мало
9,price,Nullable(Int64),35134,2.71,,1294757,0,0.00,


time: 2.09 s (started: 2026-06-01 10:04:29 +03:00)


In [8]:
end_time = time.perf_counter()
total_time = end_time - start_time
print(f"Общее время выполнения: {total_time:.4f} секунд")

Общее время выполнения: 2.7873 секунд
time: 623 μs (started: 2026-06-01 10:04:31 +03:00)
